In [1]:
import darts
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

In [2]:
import csv

input_file = "data.csv"
output_file = "single_device.csv"

device_id_to_keep = None

with open(input_file, newline="") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames)

    writer.writeheader()

    for row in reader:
        if device_id_to_keep is None:
            device_id_to_keep = row["deviceId"]
            print("Selected device:", device_id_to_keep)

        if row["deviceId"] == device_id_to_keep:
            writer.writerow(row)

Selected device: 000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe


In [27]:
devices = pd.read_csv('devices.csv')
devices

,latitude,longitude,deviceId
0,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...
1,53.5,21.1,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...
2,52.9,18.1,01668c64ccc16c506a7c1a5c032e2eb5e2de48ecb284f2...
3,52.5,17.7,01bf745bf2df0312bd5ff2234c0e9dedc39ad0bac9bcfc...
4,50.7,16.7,02e4ad5d8d0016d35a003ea6df7e10fe27093aba81c64a...
...,...,...,...
595,52.4,14.6,fbdf266dd372334fcd865972f980574608f7fc122ecf6c...
596,54.6,18.1,fc376d9b600a622120a38da673f71d121b850055851e26...
597,51.9,19.6,fd2d77f969dd205ffe471e2349d3d2f54dc6e30a7d2ccd...
598,51.5,21.5,fdd1157e24eae9617881107d653c75ee16234d3bb7d84f...


In [28]:
df = pd.read_csv('single_device.csv')


df = df.merge(devices, on="deviceId", how="left")

df["timedate"] = pd.to_datetime(df["timedate"], utc=True)

# encode time features
df["month"] = df["timedate"].dt.month
df["hour"] = df["timedate"].dt.hour

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df.drop(columns=["deviceId", "timedate", "hour", "month"], inplace=True)

# split datasets
train_df = df[df["period"] == "train"].copy()
valid_df = df[df["period"] == "valid"].copy()
test_df  = df[df["period"] == "test"].copy()

train_df.drop(columns=['period'], inplace=True)
valid_df.drop(columns=['period'], inplace=True)
test_df.drop(columns=['period'], inplace=True)

In [29]:
train_df.head()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,x1,x2,x3,deviceType,latitude,longitude,month_sin,month_cos,hour_sin,hour_cos
0,0.29,0.05,0.0,0.43,0.47,0.45,0.2,0.51,0.42,0.21,...,0.0,0.0,8,19,50.0,18.3,-0.866025,0.5,0.0,1.0
1,0.29,0.05,0.0,0.39,0.46,0.45,0.2,0.51,0.41,0.21,...,0.0,0.0,8,19,50.0,18.3,-0.866025,0.5,0.0,1.0
2,0.29,0.05,0.0,0.38,0.46,0.45,0.2,0.50,0.41,0.21,...,0.0,0.0,8,19,50.0,18.3,-0.866025,0.5,0.0,1.0
3,0.29,0.05,0.0,0.38,0.45,0.45,0.2,0.50,0.40,0.21,...,0.0,0.0,8,19,50.0,18.3,-0.866025,0.5,0.0,1.0
4,0.29,0.05,0.0,0.37,0.45,0.45,0.2,0.50,0.40,0.21,...,0.0,0.0,8,19,50.0,18.3,-0.866025,0.5,0.0,1.0


In [30]:
train_df.shape

(60789, 23)

In [31]:
split_point = int(0.8 * 60789)
train_test_df = train_df[:split_point]
train_valid_df = train_df[split_point:]

In [34]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

target = "x2"

X = train_df.drop(columns=[target])
y = train_df[target]

split_idx = int(len(train_df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Convert to DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Model parameters
params = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

# Train model
model = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=20,
)

# Predict
preds = model.predict(dtest)

# Evaluate
rmse = mean_absolute_error(y_test, preds)
print("MAE:", rmse)

[0]	train-mae:0.16245	test-mae:0.18270
[1]	train-mae:0.14776	test-mae:0.16735
[2]	train-mae:0.13478	test-mae:0.15393
[3]	train-mae:0.12380	test-mae:0.14234
[4]	train-mae:0.11303	test-mae:0.13114
[5]	train-mae:0.10360	test-mae:0.12205
[6]	train-mae:0.09530	test-mae:0.11335
[7]	train-mae:0.08772	test-mae:0.10562
[8]	train-mae:0.08114	test-mae:0.09876
[9]	train-mae:0.07557	test-mae:0.09307
[10]	train-mae:0.07076	test-mae:0.08806
[11]	train-mae:0.06661	test-mae:0.08321
[12]	train-mae:0.06314	test-mae:0.07895
[13]	train-mae:0.05978	test-mae:0.07497
[14]	train-mae:0.05673	test-mae:0.07131
[15]	train-mae:0.05399	test-mae:0.06773
[16]	train-mae:0.05181	test-mae:0.06545
[17]	train-mae:0.04938	test-mae:0.06244
[18]	train-mae:0.04739	test-mae:0.05987
[19]	train-mae:0.04597	test-mae:0.05775
[20]	train-mae:0.04449	test-mae:0.05545
[21]	train-mae:0.04295	test-mae:0.05389
[22]	train-mae:0.04151	test-mae:0.05201
[23]	train-mae:0.04022	test-mae:0.05024
[24]	train-mae:0.03910	test-mae:0.04884
[25]	train

In [17]:
train_df.describe()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,t12,t13,x1,x2,x3,deviceType,month_sin,month_cos,hour_sin,hour_cos
count,60789.000000,60789.000000,60789.0,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,60789.000000,...,60789.000000,60789.000000,60789.0,60789.000000,60789.0,60789.0,60789.000000,6.078900e+04,60789.000000,6.078900e+04
mean,0.283824,0.042482,0.0,0.422744,0.463730,0.448843,0.206733,0.519112,0.403024,0.207171,...,0.068619,0.071984,0.0,0.187953,8.0,19.0,0.258426,4.624654e-01,-0.000803,2.731307e-03
std,0.028845,0.004319,0.0,0.056188,0.033966,0.029595,0.004717,0.019955,0.031974,0.005689,...,0.004641,0.006505,0.0,0.198773,0.0,0.0,0.683479,5.022044e-01,0.707405,7.068140e-01
min,0.220000,0.040000,0.0,0.270000,0.330000,0.330000,0.180000,0.450000,0.320000,0.200000,...,0.050000,0.050000,0.0,0.000000,8.0,19.0,-0.866025,-5.000000e-01,-1.000000,-1.000000e+00
25%,0.260000,0.040000,0.0,0.370000,0.440000,0.430000,0.200000,0.500000,0.380000,0.200000,...,0.070000,0.070000,0.0,0.000000,8.0,19.0,-0.500000,6.123234e-17,-0.707107,-7.071068e-01
50%,0.280000,0.040000,0.0,0.430000,0.470000,0.450000,0.210000,0.520000,0.410000,0.210000,...,0.070000,0.070000,0.0,0.245963,8.0,19.0,0.500000,5.000000e-01,0.000000,6.123234e-17
75%,0.300000,0.040000,0.0,0.480000,0.490000,0.470000,0.210000,0.530000,0.430000,0.210000,...,0.070000,0.080000,0.0,0.317371,8.0,19.0,0.866025,8.660254e-01,0.707107,7.071068e-01
max,0.380000,0.050000,0.0,0.520000,0.550000,0.520000,0.210000,0.600000,0.450000,0.230000,...,0.080000,0.090000,0.0,0.666480,8.0,19.0,1.000000,1.000000e+00,1.000000,1.000000e+00
